In [25]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

True

In [26]:
llm = ChatGroq(
    model='openai/gpt-oss-120b',
    max_tokens=512,
    temperature=0
)
model=llm

In [27]:
parser = StrOutputParser()

In [28]:
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

In [29]:
parser2 = PydanticOutputParser(pydantic_object=Feedback)

In [31]:
prompt1 = PromptTemplate(
    template='classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instructions}',
    input_variables=['feedback'],
    partial_variables={'format_instructions':parser2.get_format_instructions()}
)

In [33]:
classifier_chain = prompt1 | model | parser2

In [34]:
prompt2 = PromptTemplate(
    template='write an appropriate response to this positive feedback \n {feedback}',
    input_variable=['feedback']
)

In [35]:
prompt3 = PromptTemplate(
    template='write an appropriate response to this negative feedback \n {feedback}',
    input_variable=['feedback']
)

In [37]:
branch_chain= RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 |model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model| parser),
    RunnableLambda(lambda x:'could not find sentiment')
)

In [38]:
chain = classifier_chain | branch_chain

In [43]:
result = chain.invoke({"feedback":"this is ewwww phone"})

In [44]:
print(result)

**Subject: We’re Sorry to Hear About Your Experience – Let’s Make It Right**

Hi [Customer’s Name],

Thank you for taking the time to share your feedback. I’m truly sorry to hear that your recent experience with us fell short of the high standards we strive to uphold.

**What went wrong?**  
From your comments, it sounds like [briefly restate the main issue(s) they mentioned – e.g., “the product arrived damaged,” “the support call took longer than expected,” “the service didn’t meet the promised timeline,” etc.]. That is certainly not the experience we want for any of our customers, and I understand how frustrating this must have been for you.

**What we’re doing about it**  
- **Immediate action:** [Explain the concrete step you’re taking right now – e.g., “We’ve already dispatched a replacement unit at no extra cost,” “Your support ticket has been escalated to a senior specialist,” “We’re processing a full refund,” etc.].  
- **Root‑cause investigation:** Our team is reviewing the pr

In [45]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
